## 1. Обнаружение пропущенных значений (NaN)

В реальных финансовых данных NaN появляются везде: выходные дни, задержки данных, делистинг.
`isnull()` / `isna()` — маска True/False. `sum()` даёт количество пропусков по колонкам.

In [ ]:
import pandas as pd
import numpy as np

# Имитируем котировки с пропусками (выходные, задержка данных)
dates = pd.date_range('2024-01-01', periods=8, freq='B')  # только рабочие дни
prices = pd.DataFrame({
    'AAPL': [185.2, np.nan, 186.0, 187.5, np.nan, np.nan, 189.1, 190.0],
    'GOOG': [140.1, 141.0, np.nan, 143.2, 144.0, 145.1, np.nan, 147.0],
    'IBM':  [160.0, 161.5, 162.0, np.nan, 163.5, 164.0, 165.2, 166.0],
}, index=dates)

print('DataFrame с пропусками:')
print(prices)

print('\nМаска пропусков (isnull):')
print(prices.isnull())

print('\nКоличество NaN по колонкам:')
print(prices.isnull().sum())

print('\n% пропусков по колонкам:')
print((prices.isnull().sum() / len(prices) * 100).round(1))

## 2. dropna — удаление строк/столбцов с пропусками

`dropna()` — убирает строки где есть хотя бы один NaN.
`thresh=N` — оставить строку если в ней не менее N непустых значений.
Для ML: обычно не удаляем строки — лучше заполнить. Удаляем фичи у которых >40-50% пропусков.

In [ ]:
import pandas as pd
import numpy as np

dates = pd.date_range('2024-01-01', periods=8, freq='B')
prices = pd.DataFrame({
    'AAPL': [185.2, np.nan, 186.0, 187.5, np.nan, np.nan, 189.1, 190.0],
    'GOOG': [140.1, 141.0, np.nan, 143.2, 144.0, 145.1, np.nan, 147.0],
    'IBM':  [160.0, 161.5, 162.0, np.nan, 163.5, 164.0, 165.2, 166.0],
}, index=dates)

print(prices, '\n')

# Оставить только строки без единого NaN
print('dropna() — строки без NaN:')
print(prices.dropna())

# Оставить строки где заполнены хотя бы 2 из 3 тикеров
print('\ndropna(thresh=2) — минимум 2 непустых значения:')
print(prices.dropna(thresh=2))

# Удаление колонок с пропусками (axis=1)
print('\ndropna(axis=1) — удалить колонки с любым NaN:')
print(prices.dropna(axis=1))

## 3. fillna — заполнение пропусков

Стратегии:
- `ffill` — предыдущее значение (для цен — самый логичный способ)
- `bfill` — следующее значение
- константа — среднее, медиана, 0
- `interpolate()` — линейная интерполяция (хорошо для временных рядов)

Для ML: заполняй до fit модели, иначе потеряешь строки.

In [ ]:
import pandas as pd
import numpy as np

dates = pd.date_range('2024-01-01', periods=8, freq='B')
prices = pd.DataFrame({
    'AAPL': [185.2, np.nan, 186.0, 187.5, np.nan, np.nan, 189.1, 190.0],
    'GOOG': [140.1, 141.0, np.nan, 143.2, 144.0, 145.1, np.nan, 147.0],
}, index=dates)

print('Оригинал:')
print(prices)

# ffill — для цен: «цена не менялась пока нет данных»
print('\nffill (предыдущее значение):')
print(prices.ffill())

# bfill — заполнить следующим значением
print('\nbfill (следующее значение):')
print(prices.bfill())

# Заполнить медианой колонки (типичный приём для ML-фичей)
print('\nЗаполнение медианой:')
print(prices.fillna(prices.median()))

# Линейная интерполяция — плавно между соседними точками
print('\nЛинейная интерполяция:')
print(prices.interpolate(method='linear'))

## 4. Дубликаты

`duplicated()` — маска дублирующихся строк (по умолчанию первое вхождение считается оригиналом).
`drop_duplicates()` — удалить дубли.
Для ML: дубликаты в обучающей выборке завышают метрики — модель «видела» эти примеры.

In [ ]:
import pandas as pd

# Сигналы от разных источников — могут дублироваться
signals = pd.DataFrame({
    'ticker': ['AAPL', 'GOOG', 'AAPL', 'MSFT', 'GOOG', 'AAPL'],
    'signal': ['buy',  'sell', 'buy',  'hold', 'sell', 'sell'],
    'source': ['ModelA', 'ModelA', 'ModelA', 'ModelB', 'ModelB', 'ModelC'],
})

print('Оригинальные сигналы:')
print(signals)

print('\nМаска дублей (по всем колонкам):')
print(signals.duplicated())

print('\nПосле drop_duplicates:')
print(signals.drop_duplicates())

# Дубли только по subset — тикер+сигнал (разные источники не важны)
print('\ndrop_duplicates по [ticker, signal]:')
print(signals.drop_duplicates(subset=['ticker', 'signal']))

## 5. Типы данных — dtypes, astype, to_numeric, to_datetime

Неправильные типы — одна из главных причин ошибок в ML-пайплайнах.
CSV часто читается как `object` (строка) вместо `float`.
Всегда проверяй `df.dtypes` после чтения файла.

In [9]:
import pandas as pd
import numpy as np

# Данные как они приходят из CSV (все строки)
raw = pd.DataFrame({
    'date':   ['2024-01-15', '2024-01-16', '2024-01-17'],
    'ticker': ['AAPL', 'AAPL', 'AAPL'],
    'close':  ['185.50', '186.20', 'N/A'],   # строки, N/A не распознан
    'volume': ['75000000', '82000000', ''],   # пустая строка вместо NaN
    'is_sp500': ['True', 'True', 'False'],    # булевы как строки
})

print(raw, '\n')
print('Типы до конвертации:')
print(raw.dtypes)
print(raw)

# Конвертация
raw['date']     = pd.to_datetime(raw['date'])
raw['close']    = pd.to_numeric(raw['close'], errors='coerce')   # N/A → NaN
raw['volume']   = pd.to_numeric(raw['volume'], errors='coerce')  # '' → NaN
raw['is_sp500'] = raw['is_sp500'].map({'True': True, 'False': False})

print('\nТипы после конвертации:')
print(raw.dtypes)
print(raw)

         date ticker   close    volume is_sp500
0  2024-01-15   AAPL  185.50  75000000     True
1  2024-01-16   AAPL  186.20  82000000     True
2  2024-01-17   AAPL     N/A              False 

Типы до конвертации:
date        object
ticker      object
close       object
volume      object
is_sp500    object
dtype: object
         date ticker   close    volume is_sp500
0  2024-01-15   AAPL  185.50  75000000     True
1  2024-01-16   AAPL  186.20  82000000     True
2  2024-01-17   AAPL     N/A              False

Типы после конвертации:
date        datetime64[ns]
ticker              object
close              float64
volume             float64
is_sp500              bool
dtype: object
        date ticker  close      volume  is_sp500
0 2024-01-15   AAPL  185.5  75000000.0      True
1 2024-01-16   AAPL  186.2  82000000.0      True
2 2024-01-17   AAPL    NaN         NaN     False


## 6. Выбросы (Outliers) — обнаружение и clip

Методы:
- `describe()` — быстрый осмотр min/max/std
- Z-score > 3 → выброс
- IQR (межквартильный размах) — устойчивее к сильным выбросам
- `clip(lower, upper)` — обрезать, не удалять

Для ML: выбросы смещают веса линейных моделей и ломают нормализацию.

In [10]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Дневные доходности с выбросами (flash crash, дивидендный гэп)
returns = pd.Series(np.random.normal(0.001, 0.015, 500))
# Добавляем искусственные выбросы
returns.iloc[50]  = 0.35   # +35% — явный выброс
returns.iloc[200] = -0.28  # -28% — явный выброс

print('Статистика доходностей:')
print(returns.describe().round(4))

# Метод IQR
Q1 = returns.quantile(0.25)
Q3 = returns.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = returns[(returns < lower) | (returns > upper)]
print(f'\nГраницы IQR: [{lower:.4f}, {upper:.4f}]')
print(f'Найдено выбросов: {len(outliers)}')
print(outliers)

# clip — обрезать до границ (не удалять строки!)
returns_clipped = returns.clip(lower=lower, upper=upper)
print(f'\nПосле clip — max: {returns_clipped.max():.4f}, min: {returns_clipped.min():.4f}')

Статистика доходностей:
count    500.0000
mean       0.0012
std        0.0249
min       -0.2800
25%       -0.0096
50%        0.0012
75%        0.0107
max        0.3500
dtype: float64

Границы IQR: [-0.0399, 0.0411]
Найдено выбросов: 6
50     0.350000
179    0.041803
200   -0.280000
209    0.058791
262   -0.047619
478    0.047183
dtype: float64

После clip — max: 0.0411, min: -0.0399


## 7. Кодирование категорий — get_dummies, map, pd.cut / pd.qcut

ML-модели работают только с числами. Категории нужно кодировать.
- `get_dummies` → One-Hot Encoding (OHE) — для номинальных (сектор, биржа)
- `map` / `replace` → Label Encoding — для бинарных или ординальных признаков
- `pd.cut` → бины по равным интервалам
- `pd.qcut` → бины по квантилям (равное количество в каждом)

⚠️ OHE с большим числом категорий раздувает матрицу — используй `drop_first=True` чтобы избежать мультиколлинеарности.

In [11]:
import pandas as pd
import numpy as np

stocks = pd.DataFrame({
    'ticker':   ['AAPL', 'GOOG', 'JPM', 'GS', 'XOM', 'CVX'],
    'sector':   ['Tech', 'Tech', 'Finance', 'Finance', 'Energy', 'Energy'],
    'rating':   ['buy', 'hold', 'sell', 'buy', 'hold', 'sell'],
    'return_1y': [0.32, 0.18, -0.05, 0.12, 0.08, -0.02],
})

print(stocks, '\n')

# One-Hot Encoding для сектора
sector_dummies = pd.get_dummies(stocks['sector'], prefix='sector', drop_first=True).astype(int)
print('OHE для sector (drop_first=True убирает мультиколлинеарность, но не теряет инфу про Energy. Alphabetical order):')
print(sector_dummies)

# Label Encoding для рейтинга (ординальный: sell < hold < buy)
rating_map = {'sell': 0, 'hold': 1, 'buy': 2}
stocks['rating_encoded'] = stocks['rating'].map(rating_map)
print('\nLabel encoding для rating:')
print(stocks[['ticker', 'rating', 'rating_encoded']])

# pd.qcut — разбить доходность на квантили (Q1-Q4)
stocks['return_quartile'] = pd.qcut(stocks['return_1y'], q=4, labels=['Q4_worst','Q3','Q2','Q1_best'])
print('\nКвантильные бины по годовой доходности:')
print(stocks[['ticker', 'return_1y', 'return_quartile']])

  ticker   sector rating  return_1y
0   AAPL     Tech    buy       0.32
1   GOOG     Tech   hold       0.18
2    JPM  Finance   sell      -0.05
3     GS  Finance    buy       0.12
4    XOM   Energy   hold       0.08
5    CVX   Energy   sell      -0.02 

OHE для sector (drop_first=True убирает мультиколлинеарность, но не теряет инфу про Energy. Alphabetical order):
   sector_Finance  sector_Tech
0               0            1
1               0            1
2               1            0
3               1            0
4               0            0
5               0            0

Label encoding для rating:
  ticker rating  rating_encoded
0   AAPL    buy               2
1   GOOG   hold               1
2    JPM   sell               0
3     GS    buy               2
4    XOM   hold               1
5    CVX   sell               0

Квантильные бины по годовой доходности:
  ticker  return_1y return_quartile
0   AAPL       0.32         Q1_best
1   GOOG       0.18         Q1_best
2    JPM      -

## 8. Нормализация / Масштабирование (без sklearn)

Два главных метода:
- **Min-Max Scaling** → диапазон [0, 1]. Чувствителен к выбросам.
- **Z-score (Standardization)** → среднее=0, std=1. Лучше для линейных моделей, нейросетей.

⚠️ Важно: `fit` статистику (mean, std, min, max) считай ТОЛЬКО на train-выборке, затем применяй к test.
Иначе — data leakage (утечка информации из будущего).

In [12]:
import pandas as pd
import numpy as np

np.random.seed(0)

# Финансовые фичи с разными масштабами — типичная проблема для ML
features = pd.DataFrame({
    'price':      np.random.uniform(50, 500, 6),      # диапазон 50-500
    'volume_m':   np.random.uniform(1, 100, 6),       # миллионы акций
    'pe_ratio':   np.random.uniform(8, 40, 6),        # мультипликатор
    'return_1y':  np.random.uniform(-0.3, 0.5, 6),    # -30% до +50%
})

print('Оригинальные фичи:')
print(features.round(2))

# Min-Max Scaling: (x - min) / (max - min)
minmax = (features - features.min()) / (features.max() - features.min())
print('\nMin-Max Scaling [0, 1]:')
print(minmax.round(3))

# Z-score: (x - mean) / std
zscore = (features - features.mean()) / features.std()
print('\nZ-score Standardization:')
print(zscore.round(3))
print('\nПроверка — среднее (≈0) и std (≈1) после z-score:')
print(pd.DataFrame({'mean': zscore.mean().round(3), 'std': zscore.std().round(3)}))

Оригинальные фичи:
    price  volume_m  pe_ratio  return_1y
0  296.97     44.32     26.18       0.32
1  371.84     89.29     37.62       0.40
2  321.24     96.40     10.27       0.48
3  295.20     38.96     10.79       0.34
4  240.64     79.38      8.65       0.07
5  340.65     53.36     34.64       0.32

Min-Max Scaling [0, 1]:
   price  volume_m  pe_ratio  return_1y
0  0.429     0.093     0.605      0.612
1  1.000     0.876     1.000      0.790
2  0.614     1.000     0.056      1.000
3  0.416     0.000     0.074      0.653
4  0.000     0.704     0.000      0.000
5  0.762     0.251     0.897      0.617

Z-score Standardization:
   price  volume_m  pe_ratio  return_1y
0 -0.315    -0.924     0.367      0.001
1  1.353     0.912     1.240      0.532
2  0.226     1.202    -0.845      1.161
3 -0.354    -1.143    -0.806      0.122
4 -1.569     0.507    -0.969     -1.831
5  0.658    -0.555     1.013      0.015

Проверка — среднее (≈0) и std (≈1) после z-score:
           mean  std
price      

## 9. Переименование колонок, reset_index, rename

Чистые имена колонок — обязательное условие для ML-пайплайнов.
Пробелы и спецсимволы в именах ломают sklearn, LightGBM, SQL.

In [ ]:
import pandas as pd

# Данные как они приходят из внешнего источника — грязные имена
df = pd.DataFrame({
    'Ticker Symbol':  ['AAPL', 'GOOG', 'IBM'],
    'Close Price $':  [185.5, 142.3, 162.0],
    '1Y Return (%)':  [32.1, 18.4, -5.2],
    'Market Cap (B)': [2900, 1800, 135],
})

print('До переименования:')
print(df.columns.tolist())

# Автоматическая очистка имён: нижний регистр, пробелы→_, убрать спецсимволы
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)  # всё кроме букв/цифр → _
    .str.strip('_')
)

print('После auto-clean:')
print(df.columns.tolist())
print(df)

# Точечное переименование
df = df.rename(columns={'ticker_symbol': 'ticker', '1y_return': 'return_1y'})
print('\nПосле rename:')
print(df)

# reset_index — когда индекс нужно превратить в колонку (например после groupby)
df_indexed = df.set_index('ticker')
df_reset = df_indexed.reset_index()
print('\nПосле reset_index:')
print(df_reset)

## 10. Merge и Concat — объединение DataFrames

- `pd.concat` — склеить по оси (строки или колонки)
- `pd.merge` / `df.merge` — SQL-подобный JOIN по ключу

Для ML: фичи обычно собираются из разных источников — merge обязателен.

In [ ]:
import pandas as pd

# Источник 1: ценовые данные
prices = pd.DataFrame({
    'ticker': ['AAPL', 'GOOG', 'JPM', 'XOM'],
    'close':  [185.5, 142.3, 198.2, 115.4],
    'volume': [75e6, 22e6, 14e6, 18e6],
})

# Источник 2: фундаментальные данные
fundamentals = pd.DataFrame({
    'ticker':    ['AAPL', 'GOOG', 'JPM', 'GS'],   # GS есть, XOM нет
    'pe_ratio':  [28.5, 22.1, 11.3, 12.8],
    'sector':    ['Tech', 'Tech', 'Finance', 'Finance'],
})

print('prices:')
print(prices)
print('\nfundamentals:')
print(fundamentals)

# INNER JOIN — только тикеры присутствующие в обоих (AAPL, GOOG, JPM)
inner = prices.merge(fundamentals, on='ticker', how='inner')
print('\nINNER JOIN (только совпадения):')
print(inner)

# LEFT JOIN — все из prices, NaN где нет в fundamentals (XOM → NaN)
left = prices.merge(fundamentals, on='ticker', how='left')
print('\nLEFT JOIN (все тикеры из prices):')
print(left)

# concat — добавить новые строки (новый день данных)
new_day = pd.DataFrame({'ticker': ['AAPL', 'GOOG'], 'close': [186.0, 143.1], 'volume': [70e6, 21e6]})
all_days = pd.concat([prices, new_day], ignore_index=True)
print('\nconcat (новые строки):')
print(all_days)

## 11. GroupBy и агрегации — ML-ready фичи

`groupby` — ключевой инструмент для создания фичей в ML:
- Средняя доходность по сектору
- Волатильность по тикеру
- Максимальный объём за период

`agg()` позволяет считать несколько метрик одновременно.

In [15]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Портфель из 5 акций, 20 дней
tickers = ['AAPL', 'GOOG', 'JPM', 'XOM', 'MSFT']
sectors = {'AAPL': 'Tech', 'GOOG': 'Tech', 'JPM': 'Finance', 'XOM': 'Energy', 'MSFT': 'Tech'}

rows = []
for t in tickers:
    for i in range(20):
        rows.append({
            'ticker': t,
            'sector': sectors[t],
            'date':   pd.Timestamp('2024-01-01') + pd.Timedelta(days=i),
            'return': np.random.normal(0.001, 0.015),
            'volume': np.random.uniform(10e6, 100e6),
        })
df = pd.DataFrame(rows)
print(df)

# Агрегация по тикеру — статистика для ML-фичей
by_ticker = df.groupby('ticker')['return'].agg(
    mean_return='mean',
    volatility='std',
    total_return='sum',
    min_return='min',
    max_return='max',
).round(4)
print('Фичи по тикеру:')
print(by_ticker)

# Агрегация по сектору
by_sector = df.groupby('sector').agg(
    avg_return=('return', 'mean'),
    avg_volume=('volume', 'mean'),
    n_observations=('return', 'count'),
).round(2)
print('\nФичи по сектору:')
print(by_sector)

# transform — добавить агрегат как колонку к оригинальному df (для нормализации внутри группы)
df['sector_avg_return'] = df.groupby('sector')['return'].transform('mean')
df['excess_return'] = df['return'] - df['sector_avg_return']  # доходность сверх среднего по сектору
print('\nExcess return (vs sector avg) — первые 6 строк:')
print(df[['ticker', 'sector', 'return', 'sector_avg_return', 'excess_return']].head(6).round(4))

   ticker sector       date    return        volume
0    AAPL   Tech 2024-01-01  0.008451  7.587945e+07
1    AAPL   Tech 2024-01-02 -0.001074  6.387926e+07
2    AAPL   Tech 2024-01-03 -0.002512  1.522753e+07
3    AAPL   Tech 2024-01-04 -0.002512  8.795585e+07
4    AAPL   Tech 2024-01-05  0.024688  1.185260e+07
..    ...    ...        ...       ...           ...
95   MSFT   Tech 2024-01-16 -0.009715  2.447272e+07
96   MSFT   Tech 2024-01-17  0.028987  6.867651e+07
97   MSFT   Tech 2024-01-18  0.008107  3.018424e+07
98   MSFT   Tech 2024-01-19 -0.013620  3.928597e+07
99   MSFT   Tech 2024-01-20  0.012806  7.718423e+07

[100 rows x 5 columns]
Фичи по тикеру:
        mean_return  volatility  total_return  min_return  max_return
ticker                                                               
AAPL         0.0001      0.0123        0.0017     -0.0277      0.0247
GOOG         0.0007      0.0109        0.0137     -0.0189      0.0169
JPM          0.0039      0.0101        0.0780     -0.011

## 12. Итоговый пайплайн — от сырых данных к ML-ready матрице

Собираем всё вместе: читаем, чистим, кодируем, нормализуем, агрегируем.
На выходе — матрица фичей готовая для `sklearn.fit()`.

In [16]:
import pandas as pd
import numpy as np

np.random.seed(0)

# --- 1. Сырые данные ---
raw = pd.DataFrame({
    'ticker':   ['AAPL', 'GOOG', 'JPM', 'XOM', 'MSFT', 'GS'],
    'sector':   ['Tech', 'Tech', 'Finance', 'Energy', 'Tech', 'Finance'],
    'close':    ['185.5', '142.3', '198.2', '115.4', '375.0', 'N/A'],   # строки, одна N/A
    'volume_m': [75.0, 22.0, np.nan, 18.0, 30.0, 5.0],                 # один NaN
    'pe_ratio': [28.5, 22.1, 11.3, 12.8, 35.0, 10.5],
    'rating':   ['buy', 'hold', 'sell', 'hold', 'buy', 'sell'],
})

print('=== 1. Сырые данные ===')
print(raw, '\n')
print(raw.describe(include='all').round(2))

# --- 2. Типы ---
raw['close'] = pd.to_numeric(raw['close'], errors='coerce')
print('\n=== 2. После to_numeric (N/A → NaN) ===')
print(raw[['ticker', 'close', 'volume_m']])

# --- 3. Заполнить NaN медианой ---
raw['close']    = raw['close'].fillna(raw['close'].median())
raw['volume_m'] = raw['volume_m'].fillna(raw['volume_m'].median())
print('\n=== 3. После fillna(median) ===')
print(raw[['ticker', 'close', 'volume_m']])

# --- 4. Кодирование ---
sector_dummies = pd.get_dummies(raw['sector'], prefix='s', drop_first=True).astype(int)
raw['rating_num'] = raw['rating'].map({'sell': 0, 'hold': 1, 'buy': 2})

# --- 5. Z-score нормализация числовых фичей ---
num_cols = ['close', 'volume_m', 'pe_ratio']
raw[num_cols] = (raw[num_cols] - raw[num_cols].mean()) / raw[num_cols].std()

# --- 6. Собрать финальную матрицу ---
X = pd.concat([raw[['ticker', 'rating_num'] + num_cols], sector_dummies], axis=1)
print('\n=== Финальная ML-матрица фичей ===')
print(X.round(3))
print(f'\nФорма: {X.shape[0]} строк × {X.shape[1]-1} фичей (без ticker)')

=== 1. Сырые данные ===
  ticker   sector  close  volume_m  pe_ratio rating
0   AAPL     Tech  185.5      75.0      28.5    buy
1   GOOG     Tech  142.3      22.0      22.1   hold
2    JPM  Finance  198.2       NaN      11.3   sell
3    XOM   Energy  115.4      18.0      12.8   hold
4   MSFT     Tech  375.0      30.0      35.0    buy
5     GS  Finance    N/A       5.0      10.5   sell 

       ticker sector  close  volume_m  pe_ratio rating
count       6      6      6      5.00      6.00      6
unique      6      3      6       NaN       NaN      3
top      AAPL   Tech  185.5       NaN       NaN    buy
freq        1      3      1       NaN       NaN      2
mean      NaN    NaN    NaN     30.00     20.03    NaN
std       NaN    NaN    NaN     26.73     10.19    NaN
min       NaN    NaN    NaN      5.00     10.50    NaN
25%       NaN    NaN    NaN     18.00     11.68    NaN
50%       NaN    NaN    NaN     22.00     17.45    NaN
75%       NaN    NaN    NaN     30.00     26.90    NaN
max  